<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/MisoTTS_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Miso TTS 8B — Highly Emotive Conversational Speech

An 8-billion-parameter text-to-speech model by Miso Labs, based on the Sesame CSM (Conversational Speech Model) architecture. Generates Mimi audio codes from text with a Llama 3.2-style backbone, supports voice continuation from prompt audio, and watermarks all outputs by default. English only.

### Quick Start
1. Connect to a GPU runtime — **L4 or A100 only** (T4 16 GB is not enough)
2. Run Steps 1–4 in order — no token, no sign-up needed
3. Open the Gradio UI link from Step 4 and start generating

| GPU | VRAM | Status |
|-----|------|--------|
|-----|------|--------|
| A100 | 40 GB | Recommended (also runs in fp32) |
|-----|------|--------|
| L4 | 24 GB | Works in bf16 (recommended tier) |
|-----|------|--------|
| T4 | 16 GB | ❌ not enough VRAM for the 8B weights |

**First run downloads ≈30–40 GB to the Hugging Face cache** (the model weights, the Mimi audio codec, the SilentCipher watermarker, and the Llama 3.2 tokenizer). Make sure you have the free disk space.

**Safety:** Generated audio is **watermarked by default** (SilentCipher) so it can be identified as AI-generated. If you deploy this model in another application, use your own private watermark key and keep it secret. The Miso Labs safety guidance prohibits impersonation, deceptive audio, fraud, and harmful content.

In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Clones the MisoTTS repo, installs the pinned deps (torch, torchaudio, moshi for the Mimi codec, silentcipher for watermarking), and Gradio.

import os, sys, subprocess
import torch

if not torch.cuda.is_available():
    raise SystemExit('No GPU detected. Connect a GPU runtime: Runtime → Change runtime type → L4 or A100')

gpu_name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'[GPU] {gpu_name} — {vram_gb:.1f} GB — sm_{cap[0]}{cap[1]}')

if vram_gb < 22:
    raise SystemExit(
        f'❌ {vram_gb:.0f} GB VRAM is not enough. MisoTTS 8B in bf16 needs ~16 GB for weights '
        'plus ~6–8 GB for the Mimi codec, SilentCipher watermarker, and KV cache. '
        'Reconnect to a runtime with ≥24 GB (L4 or A100).'
    )

REPO_DIR = '/content/MisoTTS'
REPO_URL = 'https://github.com/MisoLabsAI/MisoTTS.git'

if not os.path.isdir(REPO_DIR):
    print(f'[git] Cloning {REPO_URL} ...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    print(f'[git] {REPO_DIR} already present, skipping clone.')

print('[pip] Installing dependencies ...')
# ffmpeg for torchaudio to load m4a / mp3 / non-WAV formats
subprocess.run(['apt-get', 'install', '-y', 'ffmpeg', '-qq'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
# moshi provides the Mimi audio codec (used by MisoTTS)
# torchtune is needed by MisoTTS models.py. torchao will be mocked in Step 3.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'moshi==0.2.2', 'torchtune'], check=True)
# silentcipher is the watermarker (git-pinned in MisoTTS requirements)
subprocess.run(['pip', 'install', '-q', 'silentcipher @ git+https://github.com/SesameAILabs/silentcipher@d46d7d0893a583d8968ab3a6626e2289faec9152'], check=True)
# UI + ASR for auto-transcribing reference clips (Moonshine is tiny, runs on CPU)
subprocess.run(['pip', 'install', '-q', 'gradio==5.33.0', 'soundfile', 'librosa', 'numpy', 'tokenizers', 'huggingface_hub', 'transformers', 'tqdm==4.66.5'], check=True)

# bitsandbytes on linux (used by moshi for some layers; the repo's moshi_compat.py shims it for unquantized layers)
if sys.platform.startswith('linux'):
    subprocess.run(['pip', 'install', '-q', 'bitsandbytes'], check=False)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f'[sys.path] Added {REPO_DIR}')

os.makedirs('/content/audio_out', exist_ok=True)
os.makedirs('/content/ref_audio', exist_ok=True)
print('[Setup] /content/audio_out and /content/ref_audio ready.')

In [ ]:
# @title STEP 2 — Pre-cache Models
# @markdown Downloads the model weights and the Llama 3.2 tokenizer to the local cache. ≈30–40 GB total. The first run also downloads the Mimi codec and SilentCipher watermarker on first use.

# @markdown Reuses the Drive-cached weights from `TTS_Model_Loader.ipynb` if present,
# @markdown otherwise downloads to the default ~/.cache/huggingface (in-session only).
import pathlib
try:
    from google.colab import drive
    if not pathlib.Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive', force_remount=False)
    CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_TTS_Cache')
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault('HF_HOME', str(CACHE_ROOT / 'huggingface'))
    print(f'[Cache] using Drive at {CACHE_ROOT}')
except Exception:
    print('[Cache] Drive not available, using default ~/.cache/huggingface')


import os
from huggingface_hub import snapshot_download

REPO_ID = 'MisoLabs/MisoTTS'
print(f'[Download] {REPO_ID} ...')
snapshot_download(REPO_ID)
# Llama-3.2-1B tokenizer is used by MisoTTS for text tokenization
print('[Download] meta-llama/Llama-3.2-1B (text tokenizer) ...')
try:
    snapshot_download('meta-llama/Llama-3.2-1B', allow_patterns=['tokenizer*', 'special_tokens_map.json', '*.json'])
except Exception as e:
    print(f'[Download] Llama-3.2-1B tokenizer download failed: {e} — generator may still work if cached')
print('[Download] Done.')

In [ ]:
# @title STEP 3 — Core Functions (lazy model loading)
# @markdown Defines the inference wrappers and a CPU-only ASR helper for auto-transcribing reference clips.

import os, time
import torch
import torchaudio
import soundfile as sf

REPO_DIR = '/content/MisoTTS'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


# Colab torch 2.11 removed skip_code API used by torchao. Mock it so torchtune loads cleanly.
import types
sys.modules['torchao'] = types.ModuleType('torchao')

from generator import Segment, load_miso_8b

ASR_REPO = 'UsefulSensors/moonshine-base'
ASR_SAMPLE_RATE = 16000
_asr = None
GENERATOR = None
SAMPLE_RATE = None
MIMI_FRAME_SIZE = None
MAX_INPUT_CHARS = 1000

def load_generator():
    global GENERATOR, SAMPLE_RATE, MIMI_FRAME_SIZE
    if GENERATOR is not None:
        return GENERATOR
    print(f'[Load] MisoTTS 8B (bf16, {vram_gb:.0f} GB GPU) ...')
    t0 = time.time()
    GENERATOR = load_miso_8b(device='cuda', model_path_or_repo_id='MisoLabs/MisoTTS', dtype=torch.bfloat16)
    SAMPLE_RATE = GENERATOR.sample_rate
    MIMI_FRAME_SIZE = int(0.08 * SAMPLE_RATE)  # Miso backbone operates on 80ms audio frames (1920 @ 24kHz)
    print(f'[Load] Ready in {time.time()-t0:.1f}s — sample_rate={SAMPLE_RATE} Hz, mimi_frame={MIMI_FRAME_SIZE}')
    return GENERATOR

def _load_asr():
    global _asr
    if _asr is not None:
        return _asr
    from transformers import AutoProcessor, MoonshineForConditionalGeneration
    print(f'[Load] Moonshine ASR (CPU) ...')
    proc = AutoProcessor.from_pretrained(ASR_REPO)
    m = MoonshineForConditionalGeneration.from_pretrained(ASR_REPO).eval()
    _asr = (proc, m)
    return _asr

def transcribe_ref(path: str) -> str:
    if not path:
        return ''
    proc, m = _load_asr()
    wav, sr = torchaudio.load(path)
    wav = wav.mean(dim=0) if wav.ndim > 1 else wav
    if sr != ASR_SAMPLE_RATE:
        wav = torchaudio.functional.resample(wav, orig_freq=sr, new_freq=ASR_SAMPLE_RATE)
    inputs = proc(wav.numpy(), sampling_rate=ASR_SAMPLE_RATE, return_tensors='pt')
    with torch.no_grad():
        tokens = m.generate(**inputs)
    return proc.decode(tokens[0], skip_special_tokens=True).strip()

def _resample_to_model(audio: torch.Tensor, sr: int) -> torch.Tensor:
    audio = audio.mean(dim=0) if audio.ndim > 1 else audio
    if sr != SAMPLE_RATE:
        audio = torchaudio.functional.resample(audio, orig_freq=sr, new_freq=SAMPLE_RATE)
    return audio

def synth(text: str,
          ref_audio_path: str | None = None,
          ref_text: str = '',
          speaker_id: int = 0,
          max_length_s: float = 10.0,
          temperature: float = 0.7,
          topk: int = 50,
          seed: int = -1,
          out_name: str = 'miso.wav') -> tuple[str, int]:
    g = load_generator()
    text = (text or '').strip()
    if not text:
        raise RuntimeError('Text is empty.')
    if len(text) > MAX_INPUT_CHARS:
        raise RuntimeError(f'Text too long (>{MAX_INPUT_CHARS} characters). Try splitting it up.')

    # Defensive: keep model + codec on the right device (ZeroGPU-style, per the Space's app.py)
    g._model.to('cuda')
    g._audio_tokenizer.to('cuda')

    if seed is not None and int(seed) >= 0:
        torch.manual_seed(int(seed))

    context = []
    if ref_audio_path:
        if not (ref_text or '').strip():
            raise RuntimeError('Provide the reference transcript for voice cloning, or unhook the reference audio.')
        wav, sr = torchaudio.load(ref_audio_path)
        wav = _resample_to_model(wav, sr)
        usable = (wav.shape[-1] // MIMI_FRAME_SIZE) * MIMI_FRAME_SIZE
        if usable > 0:
            wav = wav[:usable].to('cuda')
            context = [Segment(speaker=int(speaker_id), text=ref_text.strip(), audio=wav)]

    audio = g.generate(
        text=text,
        speaker=int(speaker_id),
        context=context,
        max_audio_length_ms=float(max_length_s) * 1000.0,
        temperature=float(temperature),
        topk=int(topk),
    )

    out_path = os.path.join('/content/audio_out', out_name)
    sf.write(out_path, audio.cpu().numpy(), SAMPLE_RATE)
    return out_path, SAMPLE_RATE

print('[Core] Ready. Use Step 4 for the UI, Step 6 for quick test, Step 7 for batch.')

In [ ]:
# @title STEP 4 — Gradio UI
# @markdown Interactive web app for zero-shot TTS + voice continuation. Click the `.gradio.live` link to open.

import sys
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass
import gradio as gr

def _err(msg):
    return None, f'❌ {msg}'

def ui_synth(text, ref_audio, ref_text, auto_transcribe, speaker_id, max_length_s, temperature, topk, seed, progress=gr.Progress()):
    if not text or not text.strip():
        return _err('Type some text first.')
    try:
        progress(0.05, desc='Loading model...')
        load_generator()
        effective_ref_text = (ref_text or '').strip()
        if ref_audio is not None and auto_transcribe and not effective_ref_text:
            progress(0.2, desc='Auto-transcribing reference clip (CPU)...')
            try:
                effective_ref_text = transcribe_ref(ref_audio)
                yield None, f"""✍️ **Auto-transcribed:** "{effective_ref_text}"\n\n⏳ Generating...""".strip()
            except Exception as e:
                print(f'[ASR] failed: {e} — continuing without transcript')
                effective_ref_text = ''
        # Cloning tracks the reference much more closely at low temperature
        eff_temp = 0.4 if (ref_audio and effective_ref_text) else temperature
        progress(0.4, desc='Synthesizing...')
        path, sr = synth(
            text.strip(),
            ref_audio_path=ref_audio,
            ref_text=effective_ref_text,
            speaker_id=int(speaker_id),
            max_length_s=float(max_length_s),
            temperature=eff_temp,
            topk=int(topk),
            seed=int(seed) if seed is not None else -1,
        )
        progress(1.0, desc='Done.')
        yield path, f'✅ Generated at {sr} Hz'
    except Exception as e:
        yield None, f'❌ {e}'

def ui_transcribe(ref_audio):
    if not ref_audio:
        return ''
    try:
        return transcribe_ref(ref_audio)
    except Exception as e:
        return f'[ASR error] {e}'

with gr.Blocks(title='Miso TTS 8B', theme=gr.themes.Soft()) as demo:
    gr.Markdown(f'# Miso TTS 8B\n**Model:** `MisoLabs/MisoTTS` (8B, bf16) \u00b7 **GPU:** {gpu_name} ({vram_gb:.0f} GB)\n\nSesame CSM-style text-to-dialogue with optional voice continuation. English only. Generated audio is watermarked by default.')
    with gr.Row():
        with gr.Column():
            text = gr.Textbox(
                label='Text to synthesize',
                value='Hello from Miso. This is an eight billion parameter text to speech model, running on Google Colab.',
                lines=4,
                placeholder='Type what you want the voice to say...',
            )
            with gr.Accordion('Voice continuation (optional)', open=False):
                ref_audio = gr.Audio(
                    label='Reference audio — upload a short clip to continue in that voice',
                    type='filepath', info="10-30s of clean, single-speaker English speech. The model will continue in that voice.",
                    )
                with gr.Row():
                    auto_tx = gr.Checkbox(label='Auto-transcribe reference clip with Moonshine ASR', value=True)
                    tx_btn = gr.Button('Transcribe now', size='sm')
                ref_text = gr.Textbox(
                    label='Reference transcript (required for voice continuation; auto-filled if you enable auto-transcribe)',
                    lines=2,
                )
            with gr.Accordion('Advanced sampling', open=False):
                speaker_id = gr.Slider(0, 1, value=0, step=1, label='Speaker ID (0 or 1)', info="MisoTTS supports 2 distinct speakers. Use 0 for primary voice.")
                max_length = gr.Slider(2, 60, value=10, step=1, label='Max audio length (seconds)', info="Maximum audio length in seconds. 30s is the practical ceiling for most voices.")
                temperature = gr.Slider(0.1, 1.5, value=0.7, step=0.05, label='Temperature (auto-lowered to 0.4 when cloning a voice)', info="Lower = more deterministic, higher = more varied. 0.7 is a good default.")
                topk = gr.Slider(1, 100, value=50, step=1, label='Top-k', info="Only consider top K tokens at each step. 50 is a good default; 0 = disabled.")
                seed = gr.Number(value=-1, label='Seed (-1 = random)', precision=0)
            btn = gr.Button('Generate speech', variant='primary', size='lg')
        with gr.Column():
            output_audio = gr.Audio(label='Generated speech', type='filepath')
            status = gr.Markdown('')
    gr.Examples(
        examples=[
            ['The quick brown fox jumps over the lazy dog.', None, ''],
            ['It was the best of times, it was the worst of times.', None, ''],
            ['To be, or not to be: that is the question.', None, ''],
        ],
        inputs=[text, ref_audio, ref_text],
        # To try voice continuation, upload a 10-30s clip to the Reference audio
        # widget above and enable the 'Auto-transcribe ref' toggle. The model will
        # continue speaking in that voice.
    )
    btn.click(
        ui_synth,
        inputs=[text, ref_audio, ref_text, auto_tx, speaker_id, max_length, temperature, topk, seed],
        outputs=[output_audio, status],
    )
    tx_btn.click(ui_transcribe, inputs=[ref_audio], outputs=[ref_text])
    gr.Markdown('**Tips:** Voice continuation works best with 3–10 seconds of clean, single-speaker reference audio, and **the reference transcript is required** (the model uses it to align prosody). MisoTTS is English only. Generated audio carries a SilentCipher watermark identifying it as AI-generated.')

from IPython.display import clear_output as _clear
_clear()
demo.queue(max_size=8, concurrency_limit=2).launch(
    share=True, show_error=True, server_name="0.0.0.0", server_port=7860,
)
demo.load(lambda: "🎙️ MisoTTS 8B ready. Pick an example or upload a ref clip + enable auto-transcribe.", outputs=[status])


In [ ]:
# @title Step 5 — Keep Alive
# @markdown Prevents Colab from disconnecting during long generation sessions. Run anytime after launching the UI.

import threading, time
def _keep():
    while True:
        time.sleep(60)
        try:
            import requests
            requests.get('https://www.google.com', timeout=5)
        except Exception:
            pass
threading.Thread(target=_keep, daemon=True).start()
print('[Keep-Alive] Running. Stop cell to disable.')

In [ ]:
# @title Step 6 — Quick Test (one-off inference, no UI)
# @markdown Run a single TTS call from the notebook. Useful for scripting or quick checks.

TEXT = "The warm morning sun cast golden rays across the quiet valley, awakening the world with a gentle embrace." # @param {type:"string"}
REF_AUDIO = "" # @param {type:"string"}
REF_TEXT = "" # @param {type:"string"}
AUTO_TRANSCRIBE = True # @param {type:"boolean"}
SPEAKER_ID = 0 # @param {type:"integer"}
MAX_LENGTH_S = 10.0 # @param {type:"number"}
TEMPERATURE = 0.7 # @param {type:"number"}
TOPK = 50 # @param {type:"integer"}
SEED = -1 # @param {type:"integer"}

effective_ref_text = (REF_TEXT or '').strip()
if REF_AUDIO and AUTO_TRANSCRIBE and not effective_ref_text:
    try:
        effective_ref_text = transcribe_ref(REF_AUDIO)
        print(f'[ASR] auto-transcript: "{effective_ref_text}"')
    except Exception as e:
        print(f'[ASR] failed: {e} — continuing without transcript')
        effective_ref_text = ''

eff_temp = 0.4 if (REF_AUDIO and effective_ref_text) else TEMPERATURE
path, sr = synth(
    TEXT,
    ref_audio_path=REF_AUDIO or None,
    ref_text=effective_ref_text,
    speaker_id=SPEAKER_ID,
    max_length_s=MAX_LENGTH_S,
    temperature=eff_temp,
    topk=TOPK,
    seed=SEED,
)
print(f'[Done] {path} ({sr} Hz)')
from IPython.display import Audio, display
display(Audio(path))

In [ ]:
# @title Step 7 — Batch Synthesis
# @markdown Generate audio for a list of texts. Each line in `BATCH` becomes one output file.

BATCH_REF_AUDIO = '' # @param {type:"string"}
BATCH_REF_TEXT = '' # @param {type:"string"}
BATCH_AUTO_TRANSCRIBE = True # @param {type:"boolean"}
BATCH_SPEAKER_ID = 0 # @param {type:"integer"}
BATCH_MAX_LENGTH_S = 10.0 # @param {type:"number"}
BATCH_TEMPERATURE = 0.7 # @param {type:"number"}
BATCH_TOPK = 50 # @param {type:"integer"}
BATCH_SEED = -1 # @param {type:"integer"}

BATCH = """\
The quick brown fox jumps over the lazy dog.
She sells seashells by the seashore.
To be, or not to be: that is the question.
It was the best of times, it was the worst of times.
All happy families are alike; each unhappy family is unhappy in its own way.
"""

import shutil
lines = [l.strip() for l in BATCH.strip().splitlines() if l.strip()]
out_dir = '/content/audio_out/batch'
os.makedirs(out_dir, exist_ok=True)
effective_ref_text = (BATCH_REF_TEXT or '').strip()
if BATCH_REF_AUDIO and BATCH_AUTO_TRANSCRIBE and not effective_ref_text:
    try:
        effective_ref_text = transcribe_ref(BATCH_REF_AUDIO)
        print(f'[ASR] auto-transcript: "{effective_ref_text}"')
    except Exception as e:
        print(f'[ASR] failed: {e}')
        effective_ref_text = ''

eff_temp = 0.4 if (BATCH_REF_AUDIO and effective_ref_text) else BATCH_TEMPERATURE

for i, line in enumerate(lines):
    print(f'[{i+1}/{len(lines)}] {line[:60]}{"..." if len(line) > 60 else ""}')
    path, sr = synth(
        line,
        ref_audio_path=BATCH_REF_AUDIO or None,
        ref_text=effective_ref_text,
        speaker_id=BATCH_SPEAKER_ID,
        max_length_s=BATCH_MAX_LENGTH_S,
        temperature=eff_temp,
        topk=BATCH_TOPK,
        out_name=f'{i:03d}.wav',
    )
    shutil.move(path, os.path.join(out_dir, f'{i:03d}.wav'))

print(f'\n[Done] {len(lines)} files written to {out_dir}')